# §54: K5dfibk25 on whole DoTA-CAP (1,129 clips) + the onset figure

Rule: `core/docs/final/KIM_RUN.md` **§54**, written before any number. Upload branch `v_final_las_vad` to `Thesis-Final/kat-vad` first. **n = 1; printed, decides nothing; K5f stays the reference.**

Opens `dota_cap_eval` (560 clips) for one more row, §53's `K5dfibk25rr`, by the user's request (2026-11-25). The dev half (569) and LaGoVAD's whole-set read already exist and are not re-scored.

| step | what | GPU | estimate |
|---|---|---|---|
| 1-2 | setup, deps, tests | no | 3 min |
| 3 | preflight: checkpoint, config, bake, saved reads | no | 1 min |
| 4 | read the eval half, pool with the dev half | yes | 5-10 min |
| 5 | redraw `onset_aligned.png` (DoTA 1,129 + DADA 370) and print the table (K5f, K5dfibk25rr, LaGoVAD) | no | 3-5 min |

Outputs: `Thesis-Final/outputs/kim_cap_eval/K5dfibk25rr/`, `kim_cap_eval/REPORTS/`, figures in `kim/REPORTS/figures/train_nexar/dota_dada/k5dfibk25_lagovad_kim_vs_lagovad_s2024/` (overwrites the dev-only figure). Re-running skips what is on Drive.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import json
import os
import pickle
import shutil
import subprocess
import sys
import time
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # read-only: data, DoTA_s1_ncc, ckpts/best.ckpt (LaGoVAD released)
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'     # read-only: A2 bakes (v2 §29), V2-S caches
DRIVE_F  = '/content/drive/MyDrive/Thesis-Final'  # code, the three checkpoints and every dev read; every output of this notebook
os.environ['PROJECT_ROOT']       = DRIVE_F
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE_F}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_F}/outputs'

os.environ['REPO'] = f'{DRIVE_F}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload branch v_final_las_vad there'
assert '## 54. ' in (REPO / 'core/docs/final/KIM_RUN.md').read_text(), 'KIM_RUN.md has no §54 -- re-upload kat-vad/'
assert (REPO / 'core/tools/dota_cap_pool.py').is_file() and (REPO / 'core/tools/zs_curves.py').is_file(), 're-upload kat-vad/'
assert '--final' in (REPO / 'core/tools/protocol_b_eval.py').read_text(), 'protocol_b_eval has no --final -- re-upload kat-vad/'
assert (REPO / 'core/tools/zs_arm_table.py').is_file(), 'no zs_arm_table -- re-upload kat-vad/'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'

K5       = constants.KIM_MAIN_ARM                       # "K5": names the eval sub-folders only (eval/<set>/K5)
K5F      = constants.KIM_REFERENCE_ARM
LAG      = constants.ZS_LAGOVAD_ARM
ROW      = 'K5dfibk25' + constants.KIM_ABLATION_RERUN_SUFFIX     # §53's on row (trained on the L4)
assert 'K5dfibk25' in constants.KIM_ABLATION_ARMS
SEED     = constants.KIM_SEED                           # 2024
ENC      = constants.V5_ENCODER                         # V2-S
CAP_DEV, CAP_EVAL = constants.V2_SPLIT_DOTA_CAP_DEV, constants.V2_SPLIT_DOTA_CAP_EVAL
CAP_ALL  = constants.DOTA_CAP_ALL_DIRNAME
STRIDE   = constants.NEXAR_EVAL_STRIDE                  # protocol B: s1[::3]
N_DEV, N_EVAL = constants.DOTA_CAP_DEV_CLIPS, constants.DOTA_CAP_EVAL_CLIPS

ROW_ROOT = constants.OUTPUT_ROOT / constants.KIM_ABLATION_OUTPUT_DIRNAME / ROW
RUN_DIR  = ROW_ROOT / K5 / f's{SEED}'
DEV_READ = ROW_ROOT / 'eval' / 'dota' / K5                             # §53's DoTA-CAP-dev read
V2_CACHE = Path(DRIVE_V2) / 'cache'
A2_NEXAR = V2_CACHE / 'v2' / 'A2_S' / 'Nexar_s8'                       # the Nexar statistics the row was trained with
BAKE_NAME = f'{CAP_EVAL}_s{STRIDE}_from_s1_nexar'
BAKES    = [V2_CACHE / 'v2' / 'A2_S' / BAKE_NAME, Path(DRIVE_F) / 'cache' / 'v2' / 'A2_S' / BAKE_NAME]
DOTA     = constants.DATA_ROOT / 'DoTA'
DOTA_S1  = Path(DRIVE) / 'cache' / 'clip' / 'DoTA_s1_ncc'

CAP35    = constants.OUTPUT_ROOT / 'kim_cap_eval'       # §35: K5f, LaGoVAD on whole DoTA-CAP (read-only)
OUT      = CAP35 / ROW / 'eval'
EVAL_READ, ALL_READ = OUT / CAP_EVAL / K5, OUT / CAP_ALL / K5
REPORTS  = CAP35 / 'REPORTS'
LAG_ALL, K5F_ALL = CAP35 / LAG / 'eval' / CAP_ALL / 'B', CAP35 / K5F / 'eval' / CAP_ALL / K5
DADA_ROW = ROW_ROOT / 'eval' / 'dada' / K5 / 'clip_scores.npz'
DADA_LAG = constants.OUTPUT_ROOT / constants.ZS_LAGOVAD_OUTPUT_DIRNAME / 'eval' / 'dada' / LAG / 'clip_scores.npz'
FIGURES  = (constants.OUTPUT_ROOT / constants.KIM_OUTPUT_DIRNAME / 'REPORTS' / constants.ZS_CURVES_FIGURES_DIRNAME
            / 'train_nexar' / 'dota_dada' / 'k5dfibk25_lagovad_kim_vs_lagovad_s2024')
LOCAL    = Path('/content/k5dfibk25_cap_eval')
LOCAL.mkdir(parents=True, exist_ok=True)
print('code', REPO, '| outputs', OUT)

In [ ]:
%%bash
pip install -q "transformers==4.56.*" av einops faiss-cpu scikit-learn
python -c "import av, einops, faiss, sklearn; print('deps OK')"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU -- step 4 needs one'
df -h /content | tail -1

In [ ]:
import numpy as np
import torch
import yaml

from core.data.v2_splits import load_split           # noqa: E402
from core.tools.v5_stage1 import _flat                # noqa: E402

ENV = dict(os.environ, USE_TF='0', TF_CPP_MIN_LOG_LEVEL='2')   # transformers must not import TF


def run(cmd, log=None):
    '''Stream a child's output here (and to ``log``); raise with its tail on failure.'''
    tail = []
    sink = open(log, 'a') if log else None
    try:
        with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            for line in proc.stdout:
                tail = (tail + [line])[-40:]
                if sink:
                    sink.write(line)
                print(line, end='')
    finally:
        if sink:
            sink.close()
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd[:4])} ...')


def stage(src, dst, ids=None):
    '''Copy a Drive dir (only ``ids`` + json/npz manifests if given) to VM disk once (bit-identical; C2).'''
    t0 = time.time()
    dst.mkdir(parents=True, exist_ok=True)
    if ids is None:
        for f in src.iterdir():
            if f.is_file() and not (dst / f.name).exists():
                shutil.copy2(f, dst / f.name)
    else:
        for v in ids:
            if not (dst / f'{v}.npy').exists():
                shutil.copy2(src / f'{v}.npy', dst / f'{v}.npy')
        for pattern in ('*.json', '*.npz'):
            for m in src.glob(pattern):
                shutil.copy2(m, dst / m.name)
    print(f'  staged {src.name} in {time.time() - t0:.0f}s')
    return dst


def publish(src, dst):
    '''VM-local result -> Drive ATOMICALLY: ``<dst>.partial``, fsync, rename (C10).'''
    if src.is_dir():
        for f in [p for p in src.rglob('*') if p.is_file()]:
            publish(f, dst / f.relative_to(src))
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name(dst.name + '.partial')
    shutil.copy2(src, tmp)
    with open(tmp, 'rb+') as fh:
        os.fsync(fh.fileno())
    os.replace(tmp, dst)


def flush_drive():
    drive.flush_and_unmount()
    drive.mount('/content/drive')


def metrics_last(path):
    lines = [l for l in path.read_text().splitlines() if l.strip()] if path.exists() else []
    return json.loads(lines[-1]) if lines else None


def ckpt_step(path):
    '''The checkpoint's own global_step; 0 if absent, -1 if unreadable (never equals a real step).'''
    if not path.exists():
        return 0
    try:
        return int(torch.load(path, map_location='cpu', weights_only=False).get('global_step', 0))
    except (RuntimeError, EOFError, pickle.UnpicklingError) as exc:
        print(f'UNREADABLE {path}: {exc}')
        return -1


def manifest(d):
    return json.loads((d / constants.V2_INPUT_MANIFEST_FILENAME).read_text())


def missing_ids(d, ids):
    return [v for v in ids if not (d / f'{v}.npy').exists()]

In [ ]:
run([sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider', 'core/tests/test_dota_cap_pool.py', 'core/tests/test_zs_curves.py'])

## 3. Preflight (CPU)

In [ ]:
# ids only: the sealed split is named here on purpose (user, 2026-11-25); no score is read in this cell
EVAL_IDS, DEV_IDS = sorted(load_split(CAP_EVAL, final=True)), sorted(load_split(CAP_DEV))
assert (len(DEV_IDS), len(EVAL_IDS)) == (N_DEV, N_EVAL) and not set(DEV_IDS) & set(EVAL_IDS)
CKPT = RUN_DIR / 'checkpoint_last.pt'
STEPS = metrics_last(RUN_DIR / 'metrics.jsonl')['global_step']
assert ckpt_step(CKPT) == STEPS == json.loads((RUN_DIR / 'verified.json').read_text())['step'], 'J10'
cfg = _flat(yaml.safe_load((RUN_DIR / 'config.yaml').read_text()))
want = {'kim.enabled': True, 'kim.prototypes': True, 'kim.edge_mask': True, 'kim.seam_mask': True, 'train.seed': SEED,
        'v2.crn': constants.V2_OFF, 'v2.motion': ENC, 'dvs.theta': 0.7, 'dvs.theta_ego': 0.85, 'data.is_egocentric': False,
        'loss.dvs_anchor_mode': constants.DVS_ANCHOR_MODE_IGNORE, 'loss.bottomk_weight': constants.KIM_BOTTOMK25_WEIGHT}
wrong = {k: (cfg.get(k), v) for k, v in want.items() if cfg.get(k) != v}
assert not wrong, f'{ROW}: config (got, want) {wrong}'
for p in (DEV_READ / 'clip_scores.npz', LAG_ALL / 'clip_scores.npz', K5F_ALL / 'clip_scores.npz', DADA_ROW, DADA_LAG):
    assert p.exists(), f'missing {p}'
SHA = manifest(A2_NEXAR)['train_ids_sha1']


def bake_ok(d):
    if not (d / constants.V2_INPUT_MANIFEST_FILENAME).exists():
        return False
    m = manifest(d)
    return ((m['crn'], m['motion'], m['train_ids_sha1'], m['baked_ids']) == (constants.V2_OFF, ENC, SHA, N_EVAL)
            and not missing_ids(d, EVAL_IDS))


BAKE = next((d for d in BAKES if bake_ok(d)), None)
assert BAKE, f'no eval bake under {BAKES}: §35 read it, paste this to Claude'
assert not missing_ids(DOTA_S1, EVAL_IDS), 'CLIP s1 rows of eval clips are missing'
print(f'{ROW}: step {STEPS}, config OK | bake {BAKE}')

## 4. Read the eval half, pool

In [ ]:
from core.tools import dota_cap_pool  # noqa: E402

MARK = OUT.parent / 'OPENED.json'
if not MARK.exists():
    MARK.parent.mkdir(parents=True, exist_ok=True)
    MARK.write_text(json.dumps({'opened_utc': time.strftime('%Y-%m-%dT%H:%MZ', time.gmtime()), 'split': CAP_EVAL, 'rows': [ROW],
                                'seed': SEED, 'authorized': 'user, 2026-11-25', 'checkpoint': str(CKPT),
                                'rule': 'KIM_RUN.md §54: figure and printed table, decides nothing'}, indent=2))
if not (EVAL_READ / 'protocol_b_readout.json').exists():
    assert torch.cuda.is_available(), 'the read needs a GPU runtime'
    L_S1 = stage(DOTA_S1, LOCAL / 'stage' / 'dota_s1_eval', EVAL_IDS)
    L_IN = stage(BAKE, LOCAL / 'stage' / 'cap_eval_bake', EVAL_IDS)
    L_LAB = stage(DOTA / 'labels_s8', LOCAL / 'stage' / 'dota_labels')
    local = LOCAL / 'eval' / ROW
    shutil.rmtree(local, ignore_errors=True)
    run([sys.executable, '-m', 'core.tools.protocol_b_eval', '--run', f's{SEED}', CKPT, '--input-dir', L_IN,
         '--final', '--s1-dir', L_S1, '--metadata', DOTA / 'metadata_val.json', '--split-file', DOTA / 'val_split.txt',
         '--data-dir', L_LAB, '--split', CAP_EVAL, '--out-dir', local, '--device', 'auto'], log=LOCAL / 'eval.log')
    publish(local, EVAL_READ)
r = json.loads((EVAL_READ / 'protocol_b_readout.json').read_text())
assert (r['split'], r['clips'], r['global_steps']) == (CAP_EVAL, N_EVAL, {f's{SEED}': STEPS}), r
assert (r['arm']['crn'], r['arm']['motion']) == (constants.V2_OFF, ENC), r['arm']
if not (ALL_READ / 'protocol_b_readout.json').exists():
    local = LOCAL / 'pooled'
    shutil.rmtree(local, ignore_errors=True)
    dota_cap_pool.write_pooled(DEV_READ, EVAL_READ, local, (N_DEV, N_EVAL))
    publish(local, ALL_READ)
print('whole DoTA-CAP:', json.loads((ALL_READ / 'protocol_b_readout.json').read_text())['clips'], 'clips')
flush_drive()

## 5. Figure + table -> **paste the table to Claude**

In [ ]:
from IPython.display import Image as Show, display  # noqa: E402

from core.tools import zs_arm_table as zat  # noqa: E402

KIM_LABEL = 'LaGoVAD + KIM'
local = LOCAL / 'figures'
shutil.rmtree(local, ignore_errors=True)
run([sys.executable, '-m', 'core.tools.zs_curves', '--rank-by', KIM_LABEL, '--name', 'k5dfibk25_vs_lagovad_s2024',
     '--seed-label', f'K5dfibk25, s{SEED}, n = 1', '--pick-auc', 'global', '--set-title', 'dota', 'DoTA', '--out-dir', local,
     '--read', 'dota', KIM_LABEL, ALL_READ / 'clip_scores.npz', '--read', 'dota', LAG, LAG_ALL / 'clip_scores.npz',
     '--read', 'dada', KIM_LABEL, DADA_ROW, '--read', 'dada', LAG, DADA_LAG])
publish(local, FIGURES)
display(Show(filename=str(FIGURES / 'onset_aligned.png')))

name, label = f'cap_all_k5dfibk25_s{SEED}', 'DoTA-CAP (dev+eval)'
zat.POOLING[zat.DOTA], zat.SET_NAME[zat.DOTA] = f'{N_DEV + N_EVAL} clips, protocol B; micro / AP after per-clip min-max', label
reads = [a for row, d in ((K5F, K5F_ALL), (ROW, ALL_READ), (LAG, LAG_ALL)) for a in ('--read', zat.DOTA, row, str(d))]
zat.main([*reads, '--control', K5F, '--seed', str(SEED), '--name', name, '--out-dir', str(LOCAL / 'report')])
md_path = LOCAL / 'report' / f'{constants.ZS_ARM_TABLE_STEM}_{name}.md'
md_path.write_text(md_path.read_text().replace('DoTA-CAP-dev and DADA-2000 after', f'{label} after'))
publish(LOCAL / 'report', REPORTS)
print((REPORTS / md_path.name).read_text())
drive.flush_and_unmount()